copy the text on representation from that book and give gpt to tell u the representation for deep learning models
so u deeply understand the representation it represents
and which ml represents

Also **build projects on deep learning** too
i'm tired of chatbots , maybe just for fun or practice
but for portfolio i'm thinking nah


Optimization algorithms in deep learning are iterative methods used to update a model's internal parameters (weights and biases) to minimize a loss function, thereby improving prediction accuracy. They range from basic gradient-based methods to sophisticated adaptive techniques


## GenAI
You don't need deep mastery.

You just need:

Embeddings basics
Vector search basics
RAG architecture understanding
Cost considerations

That's enough.

You're not trying to become:

LLM engineer
GenAI specialist

You're becoming:

ML Systems Engineer

Different role.

---

remember not perfection.
70%

so ask it to define done n all that

================================


i heard a silent failure is • DataLoader num_workers misconfigured
sol: num_workers = CPU cores / 2


"Data loading not bottleneck"

They mean:

The GPU is always busy training — not waiting for data

This is a huge real-world problem.




Assumptions It Quietly Makes/ how do we check them?:
	GPU memory sufficient • Computational graph fits • Data loading not bottleneck 
    for this one: Model differentiable end-to-end, i think it is we engineers that would use our brain and think if this satisfies?

How It Fails Silently:
1.	GPU memory leak from not detaching 
2.  CPU-GPU transfer bottleneck. question: is there a way this fails in production or what?
3. • DataLoader num_workers misconfigured.
4. • Forgetting model.eval()
5. • Gradient accumulation without zero_grad()


CPU vs GPU? Batch size fitting memory? DataParallel vs DDP? Custom loss function design?

Train on GPU. Memory runs out. Debug. Optimize. Post: 'CUDA OOM on small model. Learned gradient accumulation, mixed precision, torch.no_grad(). Here's my GPU debugging workflow.'




go back to ppytorch doc
use it as reference to "Build training infrastructure and solve business problems"

Handle memory issues
Save model
Evaluate model


can create sklearn pipeline to transform the data.
split to train n validation set
is neural network sensitive to scale? i think since linear regression is, and neural network is just staked linear regression + activation then it is also sensitive to scale?

In [1]:
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


## Train on gpu

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)


Using device: cuda


## Prepare Data

In [3]:
data = pd.read_csv("../data/job_salary_prediction_dataset.csv")
data.head()

,job_title,experience_years,education_level,skills_count,industry,company_size,location,remote_work,certifications,salary
0,AI Engineer,10,Bachelor,2,Healthcare,Medium,India,Hybrid,2,109413
1,Data Analyst,5,Bachelor,17,Telecom,Small,Australia,No,0,93764
2,Frontend Developer,18,PhD,4,Media,Medium,Singapore,No,1,148123
3,Business Analyst,19,PhD,13,Retail,Medium,Canada,Yes,0,189123
4,Product Manager,15,Bachelor,7,Manufacturing,Large,Sweden,Yes,0,165069


In [4]:
# copy the feat engineering steps n paste here

data["experience_level"] = pd.cut(
    data["experience_years"],
    bins=[0, 2, 5, 10, 50],
    labels=["Junior", "Mid", "Senior", "Expert"],
)

data["skill_density"] = data["skills_count"] / (data["experience_years"] + 1)
data["cert_per_year"] = data["certifications"] / (data["experience_years"] + 1)
data["total_qualifications"] = data["skills_count"] + data["certifications"]

# Interaction features (
data["exp_x_skills"] = data["experience_years"] * data["skills_count"]
data["exp_x_cert"] = data["experience_years"] * data["certifications"]

# Premium features
data["is_tech"] = (data["industry"] == "Tech").astype(int)
data["is_masters_plus"] = (data["education_level"].isin(["Master's", "PhD"])).astype(int)


print(
    data[["experience_level", "skill_density", "total_qualifications", "is_tech"]].head()
)

  experience_level  skill_density  total_qualifications  is_tech
0           Senior       0.181818                     4        0
1              Mid       2.833333                    17        0
2           Expert       0.210526                     5        0
3           Expert       0.650000                    13        0
4           Expert       0.437500                     7        0


In [5]:
X = data.drop(["salary", "experience_level"], axis=1)  # experience_level is derived
y = data["salary"]

# Identify columns
categorical_cols = [
    "job_title",
    "education_level",
    "industry",
    "company_size",
    "location",
    "remote_work",
]
numerical_cols = [
    "experience_years",
    "skills_count",
    "certifications",
    "skill_density",
    "cert_per_year",
    "total_qualifications",
    "exp_x_skills",
    "exp_x_cert",
    "is_tech",
    "is_masters_plus",
]


In [6]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# avoids data leakage
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

y_scaler = StandardScaler()
y_train = y_scaler.fit_transform(y_train.values.reshape(-1, 1))
y_test = y_scaler.transform(y_test.values.reshape(-1, 1))


print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")


Train size: (200000, 55), Test size: (50000, 55)


In [7]:
y_train

array([[-1.28385517],
       [-0.02208821],
       [ 0.05157719],
       ...,
       [-0.39655842],
       [-1.56623028],
       [-0.33875859]], shape=(200000, 1))

In [8]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

# y_train = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)


In [9]:
y_train

tensor([[-1.2839],
        [-0.0221],
        [ 0.0516],
        ...,
        [-0.3966],
        [-1.5662],
        [-0.3388]])

In [10]:
print(torch.isnan(X_train).any())
print(torch.isinf(X_train).any())

print(torch.isnan(y_train).any())
print(torch.isinf(y_train).any())


tensor(False)
tensor(False)
tensor(False)
tensor(False)


In [11]:
y_train.shape

torch.Size([200000, 1])

## Create Dataset class

In [12]:
class SalaryDataset(Dataset):
    def __init__(self, X, y):
        self.x = X
        self.y = y

    def __len__(self):
        # tells DataLoader how many samples exist
        return len(self.x)

    def __getitem__(self, idx):
        # returns 1 sample
        return self.x[idx], self.y[idx]


In [13]:
train_dataset = SalaryDataset(X_train, y_train)
test_dataset = SalaryDataset(X_test, y_test)

In [14]:

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)

test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)


## Build Model Using nn.Module

In [15]:
class SalaryModel(nn.Module):
    def __init__(self, input_dim):
        """
        Creates a 3-layer neural network.

        Args:
        input_dim: Number of input features
        """
        super().__init__()

        # Layers in your network
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),  # Hidden Layer 1 (128 neurons)
            nn.ReLU(),
            nn.Linear(128, 64),  # Hidden Layer 2 (64 neurons)
            nn.ReLU(),
            nn.Linear(64, 1),  # Output layer (1 neuron)
        )

    def forward(self, x):
        return self.model(x)


In [16]:
torch.manual_seed(42)

# initializemodel
input_dim = X_train.shape[1]

model = SalaryModel(input_dim).to(device)

In [17]:
model

SalaryModel(
  (model): Sequential(
    (0): Linear(in_features=55, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [18]:
model.state_dict()

OrderedDict([('model.0.weight',
              tensor([[ 0.1031,  0.1119, -0.0316,  ..., -0.1034,  0.1106,  0.0388],
                      [ 0.0559,  0.0426, -0.0023,  ..., -0.0743, -0.1180, -0.0859],
                      [ 0.1348,  0.0255,  0.0416,  ...,  0.0173, -0.1188,  0.0566],
                      ...,
                      [-0.0504,  0.0309, -0.1196,  ...,  0.1186,  0.1120, -0.0629],
                      [ 0.0042,  0.0916,  0.0065,  ..., -0.0397,  0.0746,  0.0078],
                      [-0.1216, -0.1278,  0.0847,  ...,  0.0140,  0.1249, -0.1245]],
                     device='cuda:0')),
             ('model.0.bias',
              tensor([ 0.0587,  0.0236, -0.1102, -0.0979,  0.0679, -0.0674,  0.0945, -0.0932,
                       0.1019, -0.0688,  0.0546,  0.0403, -0.1058, -0.0707, -0.0327,  0.0004,
                       0.0666, -0.0554,  0.0198, -0.0731, -0.0641,  0.0880, -0.0369, -0.1182,
                       0.0057, -0.0192,  0.0383, -0.0688,  0.0058, -0.0665, -0.0815,

In [19]:
# loss function
criterion = nn.HuberLoss()

# optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

## Training Loop

In [ ]:
def train_one_epoch(dataloader, model, optimizer, criterion):

    model.train()

    total_loss = 0

    for batch_idx, (x, y) in enumerate(dataloader):
        x = x.to(device)
        y = y.to(device)

        # zero gradients
        optimizer.zero_grad()

        # forward
        predictions = model(x)

        # loss
        loss = criterion(predictions, y)

        # backward
        loss.backward()

        # step
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 10 == 0:
            print(f"Batch {batch_idx}, Loss: {loss.item()}")

    avg_loss = total_loss / len(dataloader)

    print("Training Loss:", avg_loss)
    return avg_loss


## Vaidation loop

In [21]:
def validate(dataloader, model, criterion):

    model.eval()

    total_loss = 0

    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            y = y.to(device)

            predictions = model(x)

            loss = criterion(predictions, y)

            total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)

    print("Validation Loss:", avg_loss)
    return avg_loss


## Full Training

In [22]:
epochs = 10

for epoch in range(epochs):

    train_loss = train_one_epoch(train_loader, model, optimizer, criterion)

    val_loss = validate(test_loader, model, criterion)

    print(
        f"Epoch {epoch + 1} | HuberLoss Train Loss: {train_loss:.4f} | HuberLoss Val Loss: {val_loss:.4f}"
    )


Batch 0, Loss: 0.3860672116279602
Batch 10, Loss: 0.431401789188385
Batch 20, Loss: 0.3851214349269867
Batch 30, Loss: 0.40755459666252136
Batch 40, Loss: 0.36715835332870483
Batch 50, Loss: 0.38807010650634766
Batch 60, Loss: 0.5019934773445129
Batch 70, Loss: 0.3545146882534027
Batch 80, Loss: 0.3756648600101471
Batch 90, Loss: 0.28593909740448
Batch 100, Loss: 0.31445232033729553
Batch 110, Loss: 0.37124526500701904
Batch 120, Loss: 0.29965364933013916
Batch 130, Loss: 0.34383291006088257
Batch 140, Loss: 0.38052070140838623
Batch 150, Loss: 0.3468773365020752
Batch 160, Loss: 0.37904712557792664
Batch 170, Loss: 0.309478759765625
Batch 180, Loss: 0.23995086550712585
Batch 190, Loss: 0.35123568773269653
Batch 200, Loss: 0.3052574396133423
Batch 210, Loss: 0.29763317108154297
Batch 220, Loss: 0.42596176266670227
Batch 230, Loss: 0.234395831823349
Batch 240, Loss: 0.3037857115268707
Batch 250, Loss: 0.25186580419540405
Batch 260, Loss: 0.20181970298290253
Batch 270, Loss: 0.1829441487

In [23]:
model.state_dict()

OrderedDict([('model.0.weight',
              tensor([[ 0.1020,  0.0618, -0.0267,  ..., -0.0223,  0.1292,  0.0740],
                      [ 0.0387,  0.0332,  0.0005,  ..., -0.0047, -0.0387, -0.0240],
                      [ 0.1876,  0.0361,  0.0544,  ...,  0.0407, -0.0628,  0.0523],
                      ...,
                      [-0.1000, -0.0050, -0.0719,  ...,  0.1636,  0.1393, -0.0396],
                      [ 0.0311,  0.0592, -0.0230,  ...,  0.0003,  0.0723,  0.0366],
                      [-0.1646, -0.1088,  0.0544,  ...,  0.0320,  0.1320, -0.1257]],
                     device='cuda:0')),
             ('model.0.bias',
              tensor([ 0.1288,  0.1270, -0.0770, -0.0372,  0.1687, -0.0304,  0.1530, -0.1012,
                       0.1160, -0.0427,  0.1014,  0.0505, -0.0858, -0.0759,  0.0110,  0.0607,
                       0.0958, -0.0313,  0.0660, -0.0474, -0.0467,  0.0658, -0.0224, -0.0843,
                       0.0696,  0.0114,  0.0624, -0.0087, -0.0029, -0.0379, -0.0530,

## Make Predictions

In [28]:
model.eval()

SalaryModel(
  (model): Sequential(
    (0): Linear(in_features=55, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [29]:
with torch.inference_mode():
    predictions = model(X_test.to(device))

predictions = predictions.cpu().numpy()

predictions = y_scaler.inverse_transform(predictions)
predictions[:10]

array([[170670.38],
       [ 91967.78],
       [ 76173.74],
       [167603.11],
       [115307.16],
       [169681.31],
       [112575.67],
       [ 73080.54],
       [ 97345.24],
       [186345.94]], dtype=float32)

In [ ]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test, predictions)

print("MAE:", mae)
print(f"On average, model is off by ${mae}")

MAE: 145763.578125
On average, model is off by $145763.578125


## Save and Load model

> **Architecture + weights = restored model**


In [ ]:
# If you want to resume training. Save full checkout

torch.save(
    {
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "epoch": epoch,
    },
    "checkpoint.pth",
)

# If for inference
# torch.save(model.state_dict(), "model.pth")
# then load:
# model.load_state_dict(torch.load("model.pth"))

In [32]:
input_dim

55

In [35]:
input_dim = X_train.shape[1]
input_dim

55

In [36]:
# to load you must rebuild the model architecture
class SalaryModel(nn.Module):
    def __init__(self, input_dim):
        """
        Creates a 3-layer neural network.

        Args:
        input_dim: Number of input features
        """
        super().__init__()

        # Layers in your network
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),  # Hidden Layer 1 (128 neurons)
            nn.ReLU(),
            nn.Linear(128, 64),  # Hidden Layer 2 (64 neurons)
            nn.ReLU(),
            nn.Linear(64, 1),  # Output layer (1 neuron)
        )

    def forward(self, x):
        return self.model(x)
    
model = SalaryModel(input_dim=input_dim)
model


SalaryModel(
  (model): Sequential(
    (0): Linear(in_features=55, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [37]:
model.state_dict()

OrderedDict([('model.0.weight',
              tensor([[ 0.0891,  0.0243,  0.1015,  ...,  0.1267,  0.0004, -0.0835],
                      [ 0.0015,  0.0427,  0.0527,  ...,  0.0102, -0.0135, -0.0446],
                      [ 0.0634, -0.0485, -0.0987,  ..., -0.0880, -0.1095, -0.0221],
                      ...,
                      [-0.0676, -0.0232, -0.0656,  ..., -0.0274,  0.0716,  0.0100],
                      [ 0.1201, -0.1005,  0.0041,  ..., -0.0645,  0.0256,  0.1094],
                      [-0.0422,  0.1218,  0.0719,  ..., -0.1184, -0.0256,  0.0911]])),
             ('model.0.bias',
              tensor([-0.1259,  0.0990, -0.0029, -0.0633,  0.0203,  0.0675, -0.1212, -0.1154,
                      -0.1343,  0.0802, -0.1342,  0.0836,  0.0805,  0.0280,  0.1233, -0.0238,
                      -0.0047,  0.0565, -0.0930,  0.1120, -0.0578, -0.1106,  0.0468, -0.0566,
                       0.1137, -0.0202,  0.0927, -0.0349, -0.0811, -0.0299, -0.0195,  0.0510,
                       0.024

In [39]:
# Then, load the saved weights
checkpoint = torch.load("checkpoint.pth")

# load model weights
model.load_state_dict(checkpoint["model_state"])

# load optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
optimizer.load_state_dict(checkpoint["optimizer_state"])

epoch = checkpoint["epoch"]

model.eval()

model.state_dict()

OrderedDict([('model.0.weight',
              tensor([[ 0.1020,  0.0618, -0.0267,  ..., -0.0223,  0.1292,  0.0740],
                      [ 0.0387,  0.0332,  0.0005,  ..., -0.0047, -0.0387, -0.0240],
                      [ 0.1876,  0.0361,  0.0544,  ...,  0.0407, -0.0628,  0.0523],
                      ...,
                      [-0.1000, -0.0050, -0.0719,  ...,  0.1636,  0.1393, -0.0396],
                      [ 0.0311,  0.0592, -0.0230,  ...,  0.0003,  0.0723,  0.0366],
                      [-0.1646, -0.1088,  0.0544,  ...,  0.0320,  0.1320, -0.1257]])),
             ('model.0.bias',
              tensor([ 0.1288,  0.1270, -0.0770, -0.0372,  0.1687, -0.0304,  0.1530, -0.1012,
                       0.1160, -0.0427,  0.1014,  0.0505, -0.0858, -0.0759,  0.0110,  0.0607,
                       0.0958, -0.0313,  0.0660, -0.0474, -0.0467,  0.0658, -0.0224, -0.0843,
                       0.0696,  0.0114,  0.0624, -0.0087, -0.0029, -0.0379, -0.0530,  0.0132,
                       0.043

Full Pipeline

```map
Dataset
 ↓
DataLoader
 ↓
Model(nn.Module)
 ↓
Loss
 ↓
Backward
 ↓
Optimizer
 ↓
Update
 ↓
Repeat
```